## **Import Libraries**

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier

# RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

# GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier

## **Loading the Dataset**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
data = pd.read_csv('/content/drive/MyDrive/GenAI/Churn_Prediction_Dataset/Churn Prediction.csv')

### **Exploring the Data**

In [4]:
data.shape

(10127, 21)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CLIENTNUM                 10127 non-null  int64  
 1   Attrition_Flag            10127 non-null  object 
 2   Customer_Age              10127 non-null  int64  
 3   Gender                    10127 non-null  object 
 4   Dependent_count           10127 non-null  int64  
 5   Education_Level           10127 non-null  object 
 6   Marital_Status            10127 non-null  object 
 7   Income_Category           10127 non-null  object 
 8   Card_Category             10127 non-null  object 
 9   Months_on_book            10127 non-null  int64  
 10  Total_Relationship_Count  10127 non-null  int64  
 11  Months_Inactive_12_mon    10127 non-null  int64  
 12  Contacts_Count_12_mon     10127 non-null  int64  
 13  Credit_Limit              10127 non-null  float64
 14  Total_

## **Data Processing**

In [9]:
# Dropping Clientnum
data = data.drop('CLIENTNUM', axis=1)

In [10]:
# Mapping for Attrition flag
attrition_mapping = {'Existing Customer': 0, 'Attrited Customer': 1}
data['Attrition_Flag'] = data['Attrition_Flag'].map(attrition_mapping)

In [11]:
# Create a mapping for 'Education_Level' based on hierarchy (example hierarchy)
education_mapping = {
    'Uneducated': 1,
    'High School': 2,
    'College': 3,
    'Graduate': 4,
    'Post-Graduate': 5,
    'Doctorate': 6,
    'Unknown': 0  # Handle 'Unknown' appropriately
}

data['Education_Level'] = data['Education_Level'].map(education_mapping)


# Create a mapping for 'Income_Category' based on hierarchy (example hierarchy)
income_mapping = {
    'Less than $40K': 1,
    '$40K - $60K': 2,
    '$60K - $80K': 3,
    '$80K - $120K': 4,
    '$120K +': 5,
    'Unknown': 0  # Handle 'Unknown' appropriately
}

data['Income_Category'] = data['Income_Category'].map(income_mapping)


In [12]:
# Perform 1 hot encoding for 'Marital_Status' and 'Card_Category'
data = pd.get_dummies(data, columns=['Marital_Status', 'Card_Category'],
                      drop_first=True, dtype=int)

In [14]:

# Fit and transform the 'Gender' column
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])

In [15]:
# Define the target variable (y) and the features (X)
y = data['Attrition_Flag']
X = data.drop('Attrition_Flag', axis=1)

# Split the data into training and testing sets, stratifying by the target variable
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [16]:
# Print the shapes of the resulting sets to verify the split
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (7088, 23)
X_test shape: (3039, 23)
y_train shape: (7088,)
y_test shape: (3039,)


## **Model Building**

In [17]:
# store the performance results of different models on the training and testing sets.
model_eval_results = pd.DataFrame(columns=['Model', 'Train_Accuracy', 'Train_Precision', 'Train_Recall', 'Train_F1',
                                           'Test_Accuracy', 'Test_Precision', 'Test_Recall', 'Test_F1'])


## **Model 1 : Decision Tree Classifier**

In [24]:
# Initialize Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model
dt_model.fit(X_train, y_train)

# Make predictions on training and testing data
y_train_pred = dt_model.predict(X_train)
y_test_pred = dt_model.predict(X_test)

# Evaluate the model on training data
train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)

# Evaluate the model on testing data
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

# Store the results in model_eval_results
model_eval_results.loc[len(model_eval_results)] = ['Decision Tree', train_accuracy, train_precision, train_recall, train_f1,
                                                  test_accuracy, test_precision, test_recall, test_f1]

# Show the model evaluation results dataframe
print("\nModel Evaluation Results:")
model_eval_results


Model Evaluation Results:


,Model,Train_Accuracy,Train_Precision,Train_Recall,Train_F1,Test_Accuracy,Test_Precision,Test_Recall,Test_F1
0,Decision Tree,1.000000,1.000000,1.000000,1.000000,0.933202,0.783300,0.807377,0.795156
1,Random Forest,1.000000,1.000000,1.000000,1.000000,0.955907,0.917453,0.797131,0.853070
2,Gradient Boosting,0.977003,0.956075,0.898156,0.926211,0.959197,0.929245,0.807377,0.864035
3,Decision Tree,1.000000,1.000000,1.000000,1.000000,0.933202,0.783300,0.807377,0.795156


### **Model 2 : Random Forest Classifier**

****

In [19]:
# Initialize Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions on testing data
y_test_pred_rf = rf_model.predict(X_test)

# Make predictions on training data
y_train_pred_rf = rf_model.predict(X_train)

# Evaluate the model on training data
train_accuracy_rf = accuracy_score(y_train, y_train_pred_rf)
train_precision_rf = precision_score(y_train, y_train_pred_rf)
train_recall_rf = recall_score(y_train, y_train_pred_rf)
train_f1_rf = f1_score(y_train, y_train_pred_rf)

# Evaluate the model on testing data
test_accuracy_rf = accuracy_score(y_test, y_test_pred_rf)
test_precision_rf = precision_score(y_test, y_test_pred_rf)
test_recall_rf = recall_score(y_test, y_test_pred_rf)
test_f1_rf = f1_score(y_test, y_test_pred_rf)

# Store the results in model_eval_results
model_eval_results.loc[len(model_eval_results)] = ['Random Forest', train_accuracy_rf, train_precision_rf, train_recall_rf, train_f1_rf,
                                                  test_accuracy_rf, test_precision_rf, test_recall_rf, test_f1_rf]

# Show the model evaluation results dataframe
print("\nModel Evaluation Results:")
model_eval_results


Model Evaluation Results:


,Model,Train_Accuracy,Train_Precision,Train_Recall,Train_F1,Test_Accuracy,Test_Precision,Test_Recall,Test_F1
0,Decision Tree,1.0,1.0,1.0,1.0,0.933202,0.783300,0.807377,0.795156
1,Random Forest,1.0,1.0,1.0,1.0,0.955907,0.917453,0.797131,0.853070


### **Model 2 : Gradient Boost**

In [20]:
# Initialize Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(random_state=42)

# Train the model
gb_model.fit(X_train, y_train)

# Make predictions on testing data
y_test_pred_gb = gb_model.predict(X_test)

# Make predictions on training data
y_train_pred_gb = gb_model.predict(X_train)

# Evaluate the model on training data
train_accuracy_gb = accuracy_score(y_train, y_train_pred_gb)
train_precision_gb = precision_score(y_train, y_train_pred_gb)
train_recall_gb = recall_score(y_train, y_train_pred_gb)
train_f1_gb = f1_score(y_train, y_train_pred_gb)

# Evaluate the model on testing data
test_accuracy_gb = accuracy_score(y_test, y_test_pred_gb)
test_precision_gb = precision_score(y_test, y_test_pred_gb)
test_recall_gb = recall_score(y_test, y_test_pred_gb)
test_f1_gb = f1_score(y_test, y_test_pred_gb)

# Store the results in model_eval_results
model_eval_results.loc[len(model_eval_results)] = ['Gradient Boosting', train_accuracy_gb, train_precision_gb, train_recall_gb, train_f1_gb,
                                                  test_accuracy_gb, test_precision_gb, test_recall_gb, test_f1_gb]

# Show the model evaluation results dataframe
print("\nModel Evaluation Results:")
model_eval_results


Model Evaluation Results:


,Model,Train_Accuracy,Train_Precision,Train_Recall,Train_F1,Test_Accuracy,Test_Precision,Test_Recall,Test_F1
0,Decision Tree,1.000000,1.000000,1.000000,1.000000,0.933202,0.783300,0.807377,0.795156
1,Random Forest,1.000000,1.000000,1.000000,1.000000,0.955907,0.917453,0.797131,0.853070
2,Gradient Boosting,0.977003,0.956075,0.898156,0.926211,0.959197,0.929245,0.807377,0.864035


### **Final Model Selection**

Conclusion: Gradient Boosting is the best model, as the other models (Decision Tree and Random Forest) show signs of overfitting to the training data. Gradient Boosting's performance is well-balanced across both the training and test sets, indicating better generalization to unseen data.